# RQ1 – đợt thí nghiệm cuối (GPU lớn, chạy song song)

Notebook **tự vá code trong runtime** (không cần push). Dùng chung `runs/`, `data/` trên Drive, nên mọi checkpoint cũ
(27 run gốc, E1, E3, E4, E5, E6) được tái sử dụng. Kết quả CSV ghi **thẳng vào Drive** (`{STORE}/results/final`).

| Nhóm | Thí nghiệm | Câu hỏi | Lượt |
|---|---|---|---|
| Cơ chế | **E1b** nhiễu toàn cục | Cùng một biến đổi cho mọi ảnh có gây floater? (tách "khác nhau giữa các ảnh" với "tần số thấp") | 12 |
| Cơ chế | **E4b** oracle | Bù **chính xác** (biết tham số nhiễu) có đưa floater về mức none? | 12 |
| Cơ chế | **E4b** không γ | Nhiễu chỉ gain + WB (affine biểu diễn được chính xác): affine có loại floater? | 12 |
| Cơ chế | **E4c** affine lr 1e-2 | Affine học nhanh hơn có giảm floater? (động học) | 18 |
| Gây nhiễu | **E9** khởi tạo visual hull | Kết quả có phụ thuộc khởi tạo ngẫu nhiên? (N = 8: 3 scan; N = 4: 2 scan) | 45 |
| Method | **M1** căn chỉnh dạng đóng | Căn chỉnh màu theo ảnh bằng nghiệm bình phương tối thiểu tại mỗi bước (log / tuyến tính) | 36 |
| Tương tác | **E3b** N ∈ {12, 16} | Tác động của S tiếp tục tăng theo N hay đạt cực đại? | 36 |
| Độ vững | **E8** cấu hình nhiễu 1, 2 | P1 có phụ thuộc một cấu hình nhiễu? | 12 |
| Độ vững | **E11** seed 3, 4 | Giảm dao động mức nền (gốc và affine) | 24 |
| Tổng quát | **E10** scan khác | P1 trên các scan DTU khác có sẵn trên Drive (tự phát hiện) | 9/scan |

Sau khi huấn luyện: **đánh giá lại mọi checkpoint** với bề mặt tham chiếu lọc chặt (điểm nằm trong mặt nạ ở ≥ 80% số view
nhìn thấy) và **cùng 16 view kiểm định cho mọi N ≤ 16**. Bảng kết luận cuối dựa trên đánh giá này.

Thứ tự chạy: mục 1 → 2 → 3 → 4 (huấn luyện, chạy lại được; run đã xong tự bỏ qua) → 5 (đánh giá) → 6 (phân tích).

## 1. Setup

In [ ]:
import os, subprocess, pathlib, shutil, sys
from google.colab import drive, userdata
drive.mount('/content/drive')
STORE = '/content/drive/MyDrive/faithfulrecon_store'
REPO_URL = 'github.com/Huyrua27/faithfulrecon.git'
BRANCH = 'feature/huy-dtu-multiscan'      # cùng code với các run đã có
REPO = '/content/faithfulrecon'
token = userdata.get('GH_TOKEN')
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, f'https://{token}@{REPO_URL}', REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', f'https://{token}@{REPO_URL}', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', '-f', '-B', BRANCH, 'FETCH_HEAD'], check=True)
subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)
del token
for d in ('runs', 'data'):
    p = pathlib.Path(REPO, d)
    if p.is_symlink(): p.unlink()
    elif p.exists(): shutil.rmtree(p)
    p.symlink_to(f'{STORE}/{d}')
os.chdir(REPO); sys.path.insert(0, REPO); sys.path.insert(0, f'{REPO}/scripts')
RES = f'{STORE}/results/final'; os.makedirs(RES, exist_ok=True)
!git log --oneline -1
!pip -q install -r requirements.txt scikit-learn
!nvidia-smi --query-gpu=name,memory.total --format=csv; nproc

## 2. Vá code trong runtime (chạy lại cell này sau mỗi lần khởi động lại runtime)
Gồm: `floater_metrics.py` mới (bề mặt tham chiếu lọc chặt, tập kiểm định cố định theo N), tập kiểm định cố định,
bù affine, SH bậc 3 (E5), cắt kích thước (E6), và các bản vá mới (nhiễu toàn cục, oracle, căn chỉnh dạng đóng, visual hull).
Mỗi bản vá có tự kiểm tra; cell phải in đủ `PATCH 1 OK`, `E5 patch OK`, `E6 patch OK`, `PATCH 4 OK`.

In [ ]:
%%writefile scripts/floater_metrics.py
"""Floater and geometry metrics of trained DTU base models (one row per run_dir x seed).

python scripts/floater_metrics.py --run_dirs runs/dtu_scan24_v8 runs/photo_strong_dtu_scan24_v8 \
    --seeds 0 1 2 --out results/photometric/floater_metrics.csv

Metrics (normalized units; d = distance from a Gaussian mean to the nearest reference point):
  off_003, off_01   fraction of Gaussians with d > 0.03 / d > 0.1
  front_01          fraction of Gaussians with d > 0.1 that lie IN FRONT of the reference surface
                    (median over training views of z_gauss - z_surface < -tol)
  wfloat_01         share of held-out blending weight carried by Gaussians with d > 0.1
                    (the floaters that actually render)
  acc, comp         mean distance Gaussian -> reference and reference -> Gaussian
                    (Gaussians with opacity > 0.05)
  psnr, dssim       held-out views, ORIGINAL (unperturbed) appearance
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from frecon.camera import farthest_point_indices  # noqa: E402
from frecon.data.dtu import load_dtu_views, reference_points, split_views  # noqa: E402
from frecon.gaussians import nearest_dist  # noqa: E402
from frecon.measure import heldout_errors, render_views  # noqa: E402
from frecon.render import get_rasterizer  # noqa: E402
from frecon.train import TrainConfig, Trainer  # noqa: E402


def project(P, cam):
    Xc = P @ cam.R.T + cam.t
    z = Xc[:, 2]
    u = torch.floor(cam.fx * Xc[:, 0] / z.clamp_min(1e-6) + cam.cx).long()
    v = torch.floor(cam.fy * Xc[:, 1] / z.clamp_min(1e-6) + cam.cy).long()
    ok = (z > 1e-4) & (u >= 0) & (u < cam.W) & (v >= 0) & (v < cam.H)
    return z, u.clamp(0, cam.W - 1), v.clamp(0, cam.H - 1), ok


def zbuffer(ref, cam):
    z, u, v, ok = project(ref, cam)
    zb = torch.full((cam.H * cam.W,), float("inf"), device=ref.device)
    zb = zb.scatter_reduce(0, v[ok] * cam.W + u[ok], z[ok], "amin")
    return -F.max_pool2d(-zb.view(1, 1, cam.H, cam.W), 3, 1, 1)[0, 0]   # close small holes


@torch.no_grad()
def depth_side(means, views, tol):
    diffs = []
    for cam, mask, zb in views:
        z, u, v, ok = project(means, cam)
        zs = zb[v, u]
        ok &= mask[v, u] & torch.isfinite(zs)
        d = torch.full_like(z, float("nan"))
        d[ok] = z[ok] - zs[ok]
        diffs.append(d)
    return torch.nanmedian(torch.stack(diffs, 1), 1).values   # nan where no valid view


def split_eval(cams, n_train: int, n_test: int, test_pool: int = 0):
    """Training views as in training (FPS, nested in N). With test_pool > 0 the held-out views are
    taken outside the first max(n_train, test_pool) FPS cameras, so runs with any N <= test_pool share
    the same held-out views."""
    if test_pool <= 0:
        return split_views(cams, n_train, n_test)
    centers = torch.stack([c.center for c in cams])
    tr = farthest_point_indices(centers, n_train)
    pool = farthest_point_indices(centers, max(n_train, test_pool))
    rest = [i for i in range(len(cams)) if i not in pool]
    step = max(len(rest) // max(n_test, 1), 1)
    return tr, rest[::step][:n_test]


def strict_reference(sc, cams, masks, scale_mat, frac: float, seed: int = 0) -> torch.Tensor:
    """Reference points that lie inside the object mask in at least `frac` of the views whose image
    they project into (the default crop keeps points inside the mask of >= 1 view, which retains
    table / background points). Cached next to the default cache."""
    cache = Path(sc["ref_cache"]).with_name(f"ref_points_strict{int(round(frac * 100))}.npy")
    if cache.exists():
        return torch.from_numpy(np.load(cache)).float()
    from frecon.data.blender import _read_ply_xyz
    pts_w = _read_ply_xyz(Path(sc["stl_ply"])).numpy().astype(np.float64)
    inv = np.linalg.inv(scale_mat)
    pts = (pts_w @ inv[:3, :3].T) + inv[:3, 3]
    pts = pts[np.linalg.norm(pts, axis=1) < 1.5]
    n_in, n_frame = np.zeros(len(pts), np.int32), np.zeros(len(pts), np.int32)
    for cam, m in zip(cams, masks):
        R, t = cam.R.numpy().astype(np.float64), cam.t.numpy().astype(np.float64)
        pc = pts @ R.T + t
        z = pc[:, 2]
        ok = z > 1e-6
        u = np.floor(cam.fx * pc[:, 0] / np.where(ok, z, 1) + cam.cx).astype(np.int64)
        v = np.floor(cam.fy * pc[:, 1] / np.where(ok, z, 1) + cam.cy).astype(np.int64)
        inb = ok & (u >= 0) & (u < cam.W) & (v >= 0) & (v < cam.H)
        hit = np.zeros(len(pts), dtype=bool)
        hit[inb] = m[v[inb], u[inb]]
        n_frame += inb
        n_in += hit
    keep = (n_frame > 0) & (n_in >= frac * n_frame)
    print(f"strict reference ({frac:.2f}): kept {keep.sum()}/{len(pts)} points", flush=True)
    pts = pts[keep]
    mx = sc.get("max_ref_points", 200_000)
    if len(pts) > mx:
        pts = pts[np.random.default_rng(seed).choice(len(pts), mx, replace=False)]
    pts = pts.astype(np.float32)
    np.save(cache, pts)
    return torch.from_numpy(pts)


def load_eval_scene(cfg, dev, ref_mode: str = "default", strict_frac: float = 0.8, test_pool: int = 0):
    sc = cfg["scene"]
    assert sc.get("type") == "dtu", "floater_metrics.py supports DTU scenes only"
    cams, masks, scale_mat = load_dtu_views(sc["root"], sc.get("downscale", 4))
    tr, te = split_eval(cams, sc["n_train_views"], sc["n_test_views"], test_pool)
    if ref_mode == "strict":
        ref = strict_reference(sc, cams, masks, scale_mat, strict_frac).to(dev)
    else:
        ref = reference_points(sc.get("stl_ply"), scale_mat, cams, masks, sc.get("min_mask_views", 1),
                               sc.get("max_ref_points", 200_000), sc.get("ref_cache")).to(dev)
    views = []
    for i in tr:
        cam = cams[i].to(dev)
        views.append((cam, torch.from_numpy(masks[i]).to(dev), zbuffer(ref, cam)))
    return views, [cams[i].to(dev) for i in te], ref


def tau_tag(tau: float) -> str:
    """0.03 -> '003', 0.1 -> '01', 0.15 -> '015' (keeps the historical column names)."""
    return f"{tau:g}".replace("0.", "0", 1)


@torch.no_grad()
def metrics(model, views, test_cams, ref, rasterize, bg, tol, taus=(0.03, 0.05, 0.1, 0.15)):
    """W_float(tau) = sum_{k,p} sum_{i: d_i > tau} w_ik(p) / sum_{k,p} sum_i w_ik(p) over all pixels of
    the held-out views (torch renderer: exact blending weights, contributions < 1/255 dropped)."""
    means = model.means.detach()
    opac = model.opacities.detach().reshape(-1)
    d = nearest_dist(means, ref)
    med = depth_side(means, views, tol)
    contrib = torch.zeros(model.n, device=means.device)
    for cam in test_cams:
        contrib += model.render(rasterize, cam, bg, return_contrib=True)["contrib"]
    solid = opac > 0.05
    err = heldout_errors(render_views(model, test_cams, rasterize, bg), test_cams)
    out = {"n_gaussians": model.n}
    for tau in taus:
        far = d > tau
        t = tau_tag(tau)
        out[f"off_{t}"] = float(far.float().mean())
        out[f"wfloat_{t}"] = float(contrib[far].sum() / contrib.sum().clamp_min(1e-12))
        out[f"n_off_{t}"] = int(far.sum())                      # count, not a fraction
        out[f"opsum_off_{t}"] = float(opac[far].sum())          # total opacity of far Gaussians
        out[f"contrib_off_{t}"] = float(contrib[far].sum())     # absolute held-out weight
    off01 = d > 0.1
    out.update({
        "front_01": float(((med < -tol) & off01).sum() / off01.sum().clamp_min(1)),
        "contrib_total": float(contrib.sum()),
        "acc": float(d[solid].mean()),
        "comp": float(nearest_dist(ref, means[solid]).mean()),
        "psnr": err["psnr"], "dssim": err["dssim"],
    })
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--run_dirs", nargs="+", required=True)
    ap.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
    ap.add_argument("--tol", type=float, default=0.03)
    ap.add_argument("--taus", type=float, nargs="+", default=[0.03, 0.05, 0.1, 0.15],
                    help="distance thresholds (normalized units, scene ~ unit sphere) for the sweep")
    ap.add_argument("--out", default="results/photometric/floater_metrics.csv")
    ap.add_argument("--ref_mode", choices=["default", "strict"], default="default",
                    help="strict: keep reference points inside the mask in >= --strict_frac of the views "
                         "they project into")
    ap.add_argument("--strict_frac", type=float, default=0.8)
    ap.add_argument("--test_pool", type=int, default=0,
                    help="> 0: held-out views outside the first max(N, test_pool) FPS cameras (same views "
                         "for every N <= test_pool)")
    args = ap.parse_args()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    rows, cache = [], {}
    for rd in map(Path, args.run_dirs):
        cfg = yaml.safe_load((rd / "config.yaml").read_text(encoding="utf-8"))
        rasterize = get_rasterizer(cfg.get("renderer", "torch"))
        bg = torch.ones(3, device=dev)
        key = (cfg["scene"]["root"], cfg["scene"]["n_train_views"])
        if key not in cache:   # evaluation scene ignores `photometric`: original appearance
            cache = {key: load_eval_scene(cfg, dev, args.ref_mode, args.strict_frac, args.test_pool)}
        views, test_cams, ref = cache[key]
        photo = cfg["scene"].get("photometric") or {}
        for seed in args.seeds:
            ckpt = rd / f"seed_{seed}" / "base.pt"
            if not ckpt.exists():
                print(f"skip {ckpt} (missing)")
                continue
            model = Trainer.load(ckpt, TrainConfig(**cfg["train"]), rasterize, bg, dev).model
            tc, ic = cfg.get("train", {}), cfg.get("init", {})
            row = {"run": rd.name, "scan": Path(key[0]).name, "seed": seed, "N": key[1],
                   "exposure_stops": photo.get("exposure_stops", 0.0), "wb": photo.get("wb", 0.0),
                   "gamma": photo.get("gamma", 0.0), "photo_seed": photo.get("seed", 0),
                   "photo_mode": photo.get("mode", "isp"), "appearance": tc.get("appearance", "none"),
                   "lr_appearance": tc.get("lr_appearance", 1e-3), "sh_degree": tc.get("sh_degree", 0),
                   "cap": bool(tc.get("cap_scale_after_densify", False)), "init": ic.get("type", "random"),
                   "ref_mode": args.ref_mode, "test_pool": args.test_pool}
            row.update(metrics(model, views, test_cams, ref, rasterize, bg, args.tol, args.taus))
            rows.append(row)
            print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()}, flush=True)
            del model
            if dev == "cuda":
                torch.cuda.empty_cache()
    df = pd.DataFrame(rows)
    Path(args.out).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(args.out, index=False)
    print(f"-> {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
# === PATCH 1: tập kiểm định cố định (FRECON_TEST_POOL) và bù affine theo ảnh (train.appearance) ===
from pathlib import Path

def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')

patch('frecon/data/dtu.py',
      '    rest = [i for i in range(len(cams)) if i not in train_idx]',
      '    import os   # PATCH_FIXED_TEST\n'
      '    ref_idx = farthest_point_indices(centers, max(n_train, int(os.environ.get("FRECON_TEST_POOL", 8))))\n'
      '    rest = [i for i in range(len(cams)) if i not in ref_idx]',
      'PATCH_FIXED_TEST')

t = 'frecon/train.py'
patch(t, '    log_every: int = 500\n',
      '    log_every: int = 500\n'
      '    appearance: str = "none"        # PATCH_AFFINE: affine | oracle | cf_lin | cf_log (loss only)\n'
      '    lr_appearance: float = 1e-3\n', 'PATCH_AFFINE')
patch(t, '        self._reset_stats()\n\n    # ---- statistics used by densification',
      '        self._reset_stats()\n        self.app, self.app_opt = {}, None   # PATCH_APP_INIT\n\n'
      '    # ---- statistics used by densification', 'PATCH_APP_INIT')
patch(t, '        loss = photometric_loss(out["image"], cam.image, cfg.lambda_dssim)\n'
         '        self.opt.zero_grad(set_to_none=True)\n',
      '        loss = photometric_loss(self._appearance(out["image"], cam), cam.image, cfg.lambda_dssim)\n'
      '        self.opt.zero_grad(set_to_none=True)\n'
      '        if self.app_opt is not None:   # PATCH_APP_STEP\n'
      '            self.app_opt.zero_grad(set_to_none=True)\n', 'PATCH_APP_STEP')
patch(t, '        self.opt.step()\n        with torch.no_grad():  # keep quaternions',
      '        self.opt.step()\n        if self.app_opt is not None:   # PATCH_APP_OPT\n'
      '            self.app_opt.step()\n        with torch.no_grad():  # keep quaternions', 'PATCH_APP_OPT')
patch(t, '    def fit(self, cams,',
      '    def _appearance(self, img, cam):   # PATCH_APP_FN\n'
      '        if getattr(self.cfg, "appearance", "none") != "affine":\n'
      '            return img\n'
      '        if cam.name not in self.app:\n'
      '            A = torch.nn.Parameter(torch.eye(3, device=img.device))\n'
      '            b = torch.nn.Parameter(torch.zeros(3, device=img.device))\n'
      '            self.app[cam.name] = (A, b)\n'
      '            if self.app_opt is None:\n'
      '                self.app_opt = torch.optim.Adam([A, b], lr=self.cfg.lr_appearance)\n'
      '            else:\n'
      '                self.app_opt.add_param_group({"params": [A, b]})\n'
      '        A, b = self.app[cam.name]\n'
      '        return img @ A.T + b\n\n'
      '    def fit(self, cams,', 'PATCH_APP_FN')
print('PATCH 1 OK')


# === E5 PATCH: SH bậc 1-3 (màu phụ thuộc hướng nhìn), bật bằng train.sh_degree trong config ===
from pathlib import Path

def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')

g = 'frecon/gaussians.py'
patch(g, '    def __init__(self, means, log_scales, quats, opacity_logits, color_logits):\n',
      '    def __init__(self, means, log_scales, quats, opacity_logits, color_logits, sh_rest=None):  # PATCH_SH_INIT\n', 'PATCH_SH_INIT')
patch(g, '            "color_logits": torch.nn.Parameter(color_logits.contiguous()),\n        }\n',
      '            "color_logits": torch.nn.Parameter(color_logits.contiguous()),\n        }\n'
      '        if sh_rest is not None:   # PATCH_SH_PARAM\n'
      '            self.params["sh_rest"] = torch.nn.Parameter(sh_rest.contiguous())\n', 'PATCH_SH_PARAM')
patch(g, '    def render(self, rasterize, cam, bg, features=None, opacities=None, **kw):\n'
         '        return rasterize(self.means, self.quats, self.scales,\n'
         '                         self.opacities if opacities is None else opacities,\n'
         '                         self.colors if features is None else features, cam, bg, **kw)\n',
      '    def enable_sh(self, degree: int):   # PATCH_SH_RENDER\n'
      '        if degree > 0 and "sh_rest" not in self.params:\n'
      '            k = (degree + 1) ** 2 - 1\n'
      '            self.params["sh_rest"] = torch.nn.Parameter(torch.zeros(self.n, k, 3, device=self.means.device))\n'
      '        return self\n\n'
      '    def view_colors(self, cam):\n'
      '        if "sh_rest" not in self.params:\n'
      '            return self.colors\n'
      '        d = self.means - cam.center.to(self.means)\n'
      '        d = d / d.norm(dim=-1, keepdim=True).clamp_min(1e-9)\n'
      '        B = _sh_basis(d, self.params["sh_rest"].shape[1])          # (N, K)\n'
      '        return torch.sigmoid(self.params["color_logits"] + (B[..., None] * self.params["sh_rest"]).sum(1))\n\n'
      '    def render(self, rasterize, cam, bg, features=None, opacities=None, **kw):\n'
      '        return rasterize(self.means, self.quats, self.scales,\n'
      '                         self.opacities if opacities is None else opacities,\n'
      '                         self.view_colors(cam) if features is None else features, cam, bg, **kw)\n', 'PATCH_SH_RENDER')
patch(g, '        return cls(*(sd[k] for k in PARAM_NAMES))\n',
      '        return cls(*(sd[k] for k in PARAM_NAMES), sh_rest=sd.get("sh_rest"))   # PATCH_SH_IO\n', 'PATCH_SH_IO')
patch(g, '        return GaussianModel(*(self.params[k].detach()[keep].clone() for k in PARAM_NAMES))\n',
      '        sh = self.params.get("sh_rest")   # PATCH_SH_SUBSET\n'
      '        return GaussianModel(*(self.params[k].detach()[keep].clone() for k in PARAM_NAMES),\n'
      '                             sh_rest=None if sh is None else sh.detach()[keep].clone())\n', 'PATCH_SH_SUBSET')
s = Path(g).read_text(encoding='utf-8')
if 'PATCH_SH_BASIS' not in s:
    s += '''

# PATCH_SH_BASIS: real SH basis without the DC term (constants as in 3DGS sh_utils)
def _sh_basis(d, k):
    x, y, z = d[:, 0], d[:, 1], d[:, 2]
    C1 = 0.4886025119029199
    C2 = (1.0925484305920792, -1.0925484305920792, 0.31539156525252005, -1.0925484305920792, 0.5462742152960396)
    C3 = (-0.5900435899266435, 2.890611442640554, -0.4570457994644658, 0.3731763325901154,
          -0.4570457994644658, 1.445305721320277, -0.5900435899266435)
    xx, yy, zz = x * x, y * y, z * z
    b = [-C1 * y, C1 * z, -C1 * x]
    if k > 3:
        b += [C2[0] * x * y, C2[1] * y * z, C2[2] * (2 * zz - xx - yy), C2[3] * x * z, C2[4] * (xx - yy)]
    if k > 8:
        b += [C3[0] * y * (3 * xx - yy), C3[1] * x * y * z, C3[2] * y * (4 * zz - xx - yy),
              C3[3] * z * (2 * zz - 3 * xx - 3 * yy), C3[4] * x * (4 * zz - xx - yy),
              C3[5] * z * (xx - yy), C3[6] * x * (xx - 3 * yy)]
    return torch.stack(b[:k], -1)
'''
    Path(g).write_text(s, encoding='utf-8'); print(g, ': patched (PATCH_SH_BASIS)')

t = 'frecon/train.py'
patch(t, '    log_every: int = 500\n',
      '    log_every: int = 500\n    sh_degree: int = 0   # PATCH_SH_CFG (model built with enable_sh in phase0)\n', 'PATCH_SH_CFG')
patch(t, '            [{"params": [model.params[k]], "lr": lrs[k], "name": k} for k in PARAM_NAMES], lr=0.0, eps=1e-15)',
      '            [{"params": [model.params[k]], "lr": lrs.get(k, cfg.lr_colors / 20), "name": k}   # PATCH_SH_OPT\n'
      '             for k in model.params], lr=0.0, eps=1e-15)', 'PATCH_SH_OPT')
patch(t, '                self._append({k: m.params[k].detach()[clone] for k in PARAM_NAMES})',
      '                self._append({k: m.params[k].detach()[clone] for k in m.params})   # PATCH_SH_CLONE', 'PATCH_SH_CLONE')
patch(t, '                for k in ("quats", "opacity_logits", "color_logits"):',
      '                for k in [k for k in m.params if k not in ("means", "log_scales")]:   # PATCH_SH_SPLIT', 'PATCH_SH_SPLIT')

patch('scripts/phase0.py',
      '    model = GaussianModel.random_in_box(cfg["init"]["n_points"], cfg["init"]["half_extent"], gen, dev)\n',
      '    model = GaussianModel.random_in_box(cfg["init"]["n_points"], cfg["init"]["half_extent"], gen, dev)\n'
      '    model.enable_sh(int(cfg["train"].get("sh_degree", 0)))   # PATCH_SH_PHASE0\n', 'PATCH_SH_PHASE0')

# self-test (CPU): SH0 unchanged, SH3 trains, densifies, save/load round trip
import importlib, torch, tempfile, os
import frecon.gaussians as G, frecon.train as T
importlib.reload(G); importlib.reload(T)
from frecon.render import get_rasterizer
from frecon.camera import Camera
ras, bg = get_rasterizer('torch'), torch.ones(3)
cams = [Camera(R=torch.eye(3), t=torch.tensor([0.1 * i, 0., 3.]), fx=40., fy=40., cx=16., cy=16., W=32, H=32,
               image=torch.rand(32, 32, 3) * 0.5 + 0.25, name=f'c{i}') for i in range(3)]
for deg in (0, 3):
    m = G.GaussianModel.random_in_box(400, 1.0, torch.Generator().manual_seed(0), 'cpu').enable_sh(deg)
    tr = T.Trainer(m, T.TrainConfig(iterations=30, densify_from=5, densify_until=25, densify_every=5,
                                    log_every=0, sh_degree=deg), 1.0, ras, bg)
    tr.fit(cams, iterations=30, log=None)
    p = os.path.join(tempfile.gettempdir(), f'sh{deg}.pt'); tr.save(p)
    tr2 = T.Trainer.load(p, T.TrainConfig(sh_degree=deg), ras, bg, 'cpu')
    same = torch.allclose(tr.model.render(ras, cams[0], bg)['image'], tr2.model.render(ras, cams[0], bg)['image'])
    has = 'sh_rest' in tr2.model.params
    print(f'SH{deg}: n={tr.model.n}, sh_rest={has}, reload identical={same}')
    assert same and has == (deg > 0)
print('E5 patch OK')


# === E6 PATCH: giu gioi han kich thuoc Gaussian sau pha mat do hoa (train.cap_scale_after_densify) ===
from pathlib import Path
def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')
t = 'frecon/train.py'
patch(t, '    log_every: int = 500\n',
      '    log_every: int = 500\n    cap_scale_after_densify: bool = False   # PATCH_CAP_CFG\n', 'PATCH_CAP_CFG')
patch(t, '        return float(loss)\n',
      '        if (densify and getattr(cfg, "cap_scale_after_densify", False)   # PATCH_CAP_SCALE\n'
      '                and self.step >= cfg.densify_until and self.step % cfg.densify_every == 0):\n'
      '            with torch.no_grad():\n'
      '                big = self.model.scales.max(dim=1).values > 0.1 * self.extent\n'
      '                if big.any():\n'
      '                    self.prune(~big)\n'
      '        return float(loss)\n', 'PATCH_CAP_SCALE')
# tu kiem tra (CPU)
import importlib, torch
import frecon.train as T; importlib.reload(T)
from frecon.gaussians import GaussianModel
from frecon.render import get_rasterizer
from frecon.camera import Camera
ras0, bg0 = get_rasterizer('torch'), torch.ones(3)
cams = [Camera(R=torch.eye(3), t=torch.tensor([0.1 * i, 0., 3.]), fx=40., fy=40., cx=16., cy=16., W=32, H=32,
               image=torch.rand(32, 32, 3), name=f'c{i}') for i in range(3)]
m = GaussianModel.random_in_box(300, 1.0, torch.Generator().manual_seed(0), 'cpu')
tr = T.Trainer(m, T.TrainConfig(iterations=40, densify_from=5, densify_until=20, densify_every=5, log_every=0,
                                cap_scale_after_densify=True), 5.0, ras0, bg0)
with torch.no_grad():
    tr.model.params['log_scales'][:5] = 0.0   # 5 Gaussian scale 1.0 > nguong 0.5
tr.step = 20
tr.fit(cams, iterations=10, log=None)
n_big = int((tr.model.scales.max(1).values > 0.5).sum())
assert n_big == 0 and tr.model.n > 0, n_big
print('E6 patch OK, max scale', round(float(tr.model.scales.max()), 4))


# === PATCH 4: nhiễu toàn cục (E1b), bù chính xác (oracle, E4b), căn chỉnh dạng đóng (M1), khởi tạo visual hull (E9) ===
from pathlib import Path

def patch(path, old, new, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    assert s.count(old) == 1, f'{path}: anchor not found for {tag}'
    Path(path).write_text(s.replace(old, new), encoding='utf-8'); print(f'{path}: patched ({tag})')

def append(path, code, tag):
    s = Path(path).read_text(encoding='utf-8')
    if tag in s:
        print(f'{path}: already patched ({tag})'); return
    Path(path).write_text(s + code, encoding='utf-8'); print(f'{path}: patched ({tag})')

# (a) photometric.py: mode isp | pixel | global; lưu tham số nhiễu của từng ảnh cho oracle
append('frecon/data/photometric.py', '''

# PATCH_MODES_V2 (runtime patch from colab_rq1_final.ipynb)
ORACLE = {}   # camera name -> (params, mask) of the perturbation applied to that training view


def perturb_views(cams, masks, cfg: dict):
    params = sample_params(len(cams), cfg.get("exposure_stops", 0.0), cfg.get("wb", 0.0),
                           cfg.get("gamma", 0.0), cfg.get("seed", 0))
    mode = cfg.get("mode", "isp")
    if mode == "global":   # one transform for every view: the draw with the largest exposure change
        k0 = max(range(len(params)), key=lambda k: abs(np.log2(params[k]["gain"])))
        params = [params[k0]] * len(params)
    out = []
    for k, (cam, m, p) in enumerate(zip(cams, masks, params)):
        m = torch.as_tensor(np.asarray(m), device=cam.image.device).bool()
        img = apply(cam.image, m, p)
        if mode == "pixel":   # same per-pixel magnitude, random sign -> zero-mean, no per-view component
            g = torch.Generator().manual_seed(int(cfg.get("seed", 0)) * 1000 + k)
            sign = (torch.randint(0, 2, img.shape[:2], generator=g) * 2 - 1).to(img)[..., None]
            img = torch.where(m[..., None], (cam.image + sign * (img - cam.image)).clamp(0, 1), cam.image)
        ORACLE[cam.name] = (p, m.cpu())
        out.append(replace(cam, image=img))
    return out, params
''', 'PATCH_MODES_V2')

# (b) dtu.py: giữ mặt nạ của ảnh huấn luyện cho khởi tạo visual hull
patch('frecon/data/dtu.py', '    train = [cams[i] for i in tr]\n',
      '    train = [cams[i] for i in tr]\n    LAST_TRAIN_MASKS[:] = [masks[i] for i in tr]   # PATCH_HULL_MASKS\n',
      'PATCH_HULL_MASKS')
append('frecon/data/dtu.py', '\n\nLAST_TRAIN_MASKS = []   # PATCH_HULL_GLOBAL\n', 'PATCH_HULL_GLOBAL')

# (c) train.py: appearance = oracle | cf_lin | cf_log (chỉ áp trong hàm mất mát)
t = 'frecon/train.py'
patch(t, '        if getattr(self.cfg, "appearance", "none") != "affine":\n            return img\n',
      '        mode = getattr(self.cfg, "appearance", "none")   # PATCH_APP_MODES\n'
      '        if mode == "oracle":\n'
      '            return _oracle_appearance(img, cam)\n'
      '        if mode in ("cf_lin", "cf_log"):\n'
      '            return _closed_form_appearance(img, cam.image, mode)\n'
      '        if mode != "affine":\n'
      '            return img\n', 'PATCH_APP_MODES')
append(t, '''

# PATCH_APP_FUNCS (runtime patch from colab_rq1_final.ipynb)
def _oracle_appearance(img, cam):
    """Exact forward model of the perturbation of this training view (known parameters)."""
    from frecon.data.photometric import ORACLE
    if cam.name not in ORACLE:
        return img
    p, m = ORACLE[cam.name]
    m = m.to(img.device)
    scale = p["gain"] * torch.tensor(p["wb"], dtype=img.dtype, device=img.device)
    out = (img * scale).clamp(1e-4, 1.0) ** p["gamma"]
    return torch.where(m[..., None], out, img)


def _closed_form_appearance(img, target, mode):
    """Per-view, per-channel affine alignment of the render to the target, solved in closed form at
    every step (least squares, no gradient through the coefficients). cf_log works on log intensity,
    where gain, white balance and gamma of the ISP model are exactly affine per channel."""
    sel = (target < 0.99).any(-1)          # object pixels (background is composited white)
    if int(sel.sum()) < 64:
        return img
    if mode == "cf_log":
        x, y = img.clamp_min(1e-3).log(), target.clamp_min(1e-3).log()
    else:
        x, y = img, target
    with torch.no_grad():
        xs, ys = x[sel], y[sel]
        xm, ym = xs.mean(0), ys.mean(0)
        xc = xs - xm
        a = ((xc * (ys - ym)).mean(0) / xc.pow(2).mean(0).clamp_min(1e-6)).clamp(0.25, 4.0)
        b = ym - a * xm
    out = a * x + b
    if mode == "cf_log":
        out = out.exp()
    return torch.where(sel[..., None], out, img)
''', 'PATCH_APP_FUNCS')

# (d) phase0.py: init.type = hull (điểm ngẫu nhiên trong visual hull của mặt nạ huấn luyện)
ph0 = 'scripts/phase0.py'
patch(ph0, '    model = GaussianModel.random_in_box(cfg["init"]["n_points"], cfg["init"]["half_extent"], gen, dev)\n',
      '    model = (_hull_init(cfg, train_cams, gen, dev) if cfg["init"].get("type") == "hull" else   # PATCH_HULL\n'
      '             GaussianModel.random_in_box(cfg["init"]["n_points"], cfg["init"]["half_extent"], gen, dev))\n',
      'PATCH_HULL')
patch(ph0, 'def train_base(cfg, seed, train_cams, rasterize, bg, dev, path: Path):\n',
      '''def _hull_init(cfg, cams, gen, dev):   # PATCH_HULL_FN
    """Random points inside the visual hull of the training masks: inside the mask of every training
    view that sees them, and inside the image of at least half of the views."""
    import frecon.data.dtu as D
    masks = [torch.as_tensor(m, device=dev) for m in D.LAST_TRAIN_MASKS]
    assert len(masks) == len(cams), "hull init needs the DTU loader (masks of the training views)"
    n, h = cfg["init"]["n_points"], cfg["init"]["half_extent"]
    kept, total = [], 0
    for _ in range(200):
        P = (torch.rand(200_000, 3, generator=gen, device=dev) * 2 - 1) * h
        ok = torch.ones(len(P), dtype=torch.bool, device=dev)
        n_frame = torch.zeros(len(P), device=dev)
        for cam, m in zip(cams, masks):
            Xc = P @ cam.R.to(dev).T + cam.t.to(dev)
            z = Xc[:, 2]
            u = torch.floor(cam.fx * Xc[:, 0] / z.clamp_min(1e-6) + cam.cx).long()
            v = torch.floor(cam.fy * Xc[:, 1] / z.clamp_min(1e-6) + cam.cy).long()
            inb = (z > 1e-6) & (u >= 0) & (u < cam.W) & (v >= 0) & (v < cam.H)
            hit = torch.zeros_like(ok)
            hit[inb] = m[v[inb], u[inb]]
            ok &= hit | ~inb
            n_frame += inb
        sel = P[ok & (n_frame >= max(2, len(cams) // 2))]
        kept.append(sel)
        total += len(sel)
        if total >= n:
            break
    P = torch.cat(kept)[:n]
    log(f"hull init: {len(P)} points")
    return GaussianModel.from_points(P, torch.rand(len(P), 3, generator=gen, device=dev))


def train_base(cfg, seed, train_cams, rasterize, bg, dev, path: Path):
''', 'PATCH_HULL_FN')

# ---- tự kiểm tra (CPU, vài giây) ----
import importlib, sys, torch, numpy as np
import frecon.data.photometric as ph, frecon.data.dtu as dtu, frecon.train as T, frecon.gaussians as G
for mod in (ph, dtu, G, T):
    importlib.reload(mod)
sys.path.insert(0, 'scripts')
import phase0; importlib.reload(phase0)
from frecon.camera import Camera
from frecon.render import get_rasterizer
ras, bg = get_rasterizer('torch'), torch.ones(3)
def look(i):   # camera on a circle looking at the origin
    a = 2 * np.pi * i / 4
    c = torch.tensor([3 * np.sin(a), 0., 3 * np.cos(a)], dtype=torch.float32)
    f = -c / c.norm(); r = torch.cross(torch.tensor([0., 1., 0.]), f, dim=0); r = r / r.norm()
    R = torch.stack([r, torch.cross(f, r, dim=0), f]); return R, -R @ c
cams, masks = [], []
for i in range(4):
    R, t = look(i)
    img = torch.ones(32, 32, 3); img[8:24, 8:24] = torch.tensor([0.6, 0.4, 0.3])
    m = np.zeros((32, 32), bool); m[8:24, 8:24] = True
    cams.append(Camera(R=R, t=t, fx=40., fy=40., cx=16., cy=16., W=32, H=32, image=img, name=f'c{i}')); masks.append(m)
cfgp = dict(exposure_stops=1.0, wb=0.15, gamma=0.2, seed=0)
glob_, _ = ph.perturb_views(cams, masks, dict(cfgp, mode='global'))
assert all(torch.equal(glob_[0].image, c.image) for c in glob_), 'global mode must give identical transforms'
isp, _ = ph.perturb_views(cams, masks, cfgp)
orc = T._oracle_appearance(torch.stack([c.image for c in cams])[0], cams[0])
assert torch.allclose(orc, isp[0].image, atol=1e-3), 'oracle must reproduce the perturbed image'
for mode in ('cf_lin', 'cf_log'):
    al = T._closed_form_appearance(cams[0].image, isp[0].image, mode)
    assert (al - isp[0].image).abs().max() < 0.02, (mode, float((al - isp[0].image).abs().max()))
for app in ('oracle', 'cf_log', 'cf_lin', 'affine'):
    m0 = G.GaussianModel.random_in_box(300, 1.0, torch.Generator().manual_seed(0), 'cpu')
    tr = T.Trainer(m0, T.TrainConfig(iterations=20, densify_from=5, densify_until=15, densify_every=5,
                                     log_every=0, appearance=app), 1.0, ras, bg)
    tr.fit(isp, iterations=20, log=None)
    print(f'{app}: 20 steps OK, n={tr.model.n}')
dtu.LAST_TRAIN_MASKS[:] = masks
hm = phase0._hull_init({'init': {'n_points': 500, 'half_extent': 1.0}}, cams, torch.Generator().manual_seed(0), 'cpu')
assert hm.n == 500 and float(hm.means.abs().max()) < 0.8, float(hm.means.abs().max())
print('PATCH 4 OK (global, oracle, closed-form, hull)')

## 3. Cấu hình thí nghiệm

In [ ]:
import yaml, json, time, glob, torch, numpy as np, pandas as pd
from pathlib import Path
SEEDS = (0, 1, 2)
LEVELS = {'none': None,
          'mild':     dict(exposure_stops=0.5, wb=0.05, gamma=0.10, seed=0),
          'strong':   dict(exposure_stops=1.0, wb=0.15, gamma=0.20, seed=0),
          'strongng': dict(exposure_stops=1.0, wb=0.15, gamma=0.0, seed=0),    # không có gamma
          'strongs1': dict(exposure_stops=1.0, wb=0.15, gamma=0.20, seed=1),   # cấu hình nhiễu khác
          'strongs2': dict(exposure_stops=1.0, wb=0.15, gamma=0.20, seed=2)}
L3 = ('none', 'mild', 'strong')
S2 = (63, 83)
template = Path('configs/dtu_scan24_v8.yaml').read_text(encoding='utf-8')
APP_TAG = {'affine': '_aff', 'oracle': '_orc', 'cf_log': '_cflog', 'cf_lin': '_cflin'}

def make_cfg(s, N, level, mode=None, appearance=None, lr_app=None, init=None):
    tag = {'pixel': '_pix', 'global': '_glob', None: ''}[mode] + APP_TAG.get(appearance, '')
    tag += '10' if lr_app == 1e-2 else ''
    tag += '_hull' if init == 'hull' else ''
    base = yaml.safe_load(template.replace('scan24', f'scan{s}').replace('stl024', f'stl{s:03d}'))
    cfg = dict(base, out_dir=f'runs/photo_{level}{tag}_dtu_scan{s}_v{N}', seeds=list(SEEDS))
    cfg['scene'] = dict(base['scene'], n_train_views=N)
    if LEVELS[level]:
        cfg['scene']['photometric'] = dict(LEVELS[level], **({'mode': mode} if mode else {}))
    cfg['train'] = dict(base['train'])
    if appearance:
        cfg['train']['appearance'] = appearance
    if lr_app:
        cfg['train']['lr_appearance'] = lr_app
    if init:
        cfg['init'] = dict(base['init'], type=init)
    path = f"configs/{Path(cfg['out_dir']).name}.yaml"
    yaml.safe_dump(cfg, open(path, 'w', encoding='utf-8'), sort_keys=False)
    return path

# scan DTU khác có đủ dữ liệu trên Drive (ảnh, mặt nạ, camera, STL)
def scan_ok(s):
    r = Path(f'data/dtu/scan{s}')
    return ((r / 'cameras_sphere.npz').exists() or (r / 'cameras.npz').exists()) and (r / 'image').is_dir() \
        and (r / 'mask').is_dir() and Path(f'data/dtu/Points/stl/stl{s:03d}_total.ply').exists()
EXTRA = [s for s in (37, 40, 55, 65, 69, 97, 105, 106, 110, 114, 118, 122) if scan_ok(s)][:4]
print('scan khác có sẵn:', EXTRA or 'không có (E10 bỏ qua)')

def J(paths, seeds=SEEDS):
    return [(p, k) for p in paths for k in seeds]

EXPS = {
    'E1b_global':   J([make_cfg(s, 8, l, mode='global') for s in S2 for l in ('mild', 'strong')]),
    'E4b_oracle':   J([make_cfg(s, 8, l, appearance='oracle') for s in S2 for l in ('mild', 'strong')]),
    'E4b_nogamma':  J([make_cfg(s, 8, 'strongng', appearance=a) for s in S2 for a in (None, 'affine')]),
    'E4c_affine_lr': J([make_cfg(s, 8, l, appearance='affine', lr_app=1e-2) for s in S2 for l in L3]),
    'E9_hull':      J([make_cfg(s, 8, l, init='hull') for s in (24, 63, 83) for l in L3]
                      + [make_cfg(s, 4, l, init='hull') for s in S2 for l in L3]),
    'M1_closed_form': J([make_cfg(s, 8, l, appearance=a) for a in ('cf_log', 'cf_lin') for s in S2 for l in L3]),
    'E3b_views':    J([make_cfg(s, N, l) for N in (12, 16) for s in S2 for l in L3]),
    'E8_noise':     J([make_cfg(s, 8, l) for l in ('strongs1', 'strongs2') for s in S2]),
    'E11_seeds':    J([make_cfg(s, 8, l, appearance=a) for a in (None, 'affine') for s in S2 for l in L3], (3, 4)),
    'E10_scans':    J([make_cfg(s, 8, l) for s in EXTRA for l in L3]),
}
def out_dir(p): return yaml.safe_load(open(p, encoding='utf-8'))['out_dir']
def is_done(p, k): return os.path.exists(f'{out_dir(p)}/seed_{k}/base.pt')
for e, jobs in EXPS.items():
    print(f'{e:15s} {sum(is_done(*j) for j in jobs):3d}/{len(jobs)} đã có')

## 4. Huấn luyện song song (chạy lại được)

`WORKERS` tự chọn theo bộ nhớ GPU còn trống (khoảng 10 GB mỗi tiến trình; lượt strong từng vượt 8 GB trên L4) và số CPU. Mỗi lượt huấn luyện là một tiến trình
`phase0.py --base_only` với một seed; log ở `runs/<run>/seed_<k>/train.log`. Nếu runtime bị ngắt, chạy lại mục 1, 2, 3, 4:
các lượt đã có `base.pt` được bỏ qua.

Có thể chạy riêng từng nhóm bằng cách sửa `RUN`. Thứ tự mặc định ưu tiên các thí nghiệm quyết định kết luận.

In [ ]:
RUN = ['E1b_global', 'E4b_oracle', 'E4b_nogamma', 'E4c_affine_lr', 'E9_hull', 'M1_closed_form',
       'E3b_views', 'E8_noise', 'E11_seeds', 'E10_scans']
free_gb = torch.cuda.mem_get_info()[0] / 2**30
WORKERS = max(1, min(int(free_gb // 10), (os.cpu_count() or 8) - 1, 16))
print(f'GPU trống {free_gb:.0f} GB, CPU {os.cpu_count()} -> WORKERS = {WORKERS}')

def run_jobs(jobs, workers):
    env = dict(os.environ, PYTHONIOENCODING='utf-8', OMP_NUM_THREADS='2',
               PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')
    pending = [j for j in jobs if not is_done(*j)]
    print(f'{len(pending)} lượt cần chạy, {len(jobs) - len(pending)} đã có', flush=True)
    running, failed, t0 = [], [], time.time()
    while pending or running:
        while pending and len(running) < workers:
            p, k = pending.pop(0)
            d = f'{out_dir(p)}/seed_{k}'; os.makedirs(d, exist_ok=True)
            log = open(f'{d}/train.log', 'a')
            pr = subprocess.Popen([sys.executable, '-W', 'ignore', 'scripts/phase0.py', '--config', p,
                                   '--seeds', str(k), '--base_only'], stdout=log, stderr=subprocess.STDOUT, env=env)
            running.append((pr, p, k, log))
        time.sleep(15)
        for item in list(running):
            pr, p, k, log = item
            if pr.poll() is None:
                continue
            log.close(); running.remove(item)
            ok = pr.returncode == 0 and is_done(p, k)
            if not ok:
                failed.append((p, k))
            print(f'[{(time.time() - t0) / 60:6.1f} phút] {"OK  " if ok else "LỖI"} {Path(p).stem} seed {k} '
                  f'| chờ {len(pending)}, đang chạy {len(running)}', flush=True)
    print(f'xong sau {(time.time() - t0) / 60:.0f} phút; lỗi: {failed or "không"}')
    return failed

failed = run_jobs([j for e in RUN for j in EXPS[e]], WORKERS)

In [ ]:
# Tiến độ (chạy trong lúc chờ, hoặc từ terminal: tail -n 3 runs/*/seed_*/train.log)
for e, jobs in EXPS.items():
    print(f'{e:15s} {sum(is_done(*j) for j in jobs):3d}/{len(jobs)}')
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv

## 5. Đánh giá lại toàn bộ checkpoint (bề mặt tham chiếu lọc chặt, cùng 16 view kiểm định cho mọi N ≤ 16)

Bước 1 tạo bề mặt tham chiếu lọc chặt cho từng scan (một lần, CPU). Bước 2 chạy `floater_metrics.py` song song theo nhóm
(scan, N). Thêm một lượt đánh giá với bề mặt tham chiếu cũ trên các run gốc N = 8 để so sánh ảnh hưởng của việc lọc.

In [ ]:
from floater_metrics import load_eval_scene
dirs = sorted(d for d in glob.glob('runs/photo_*_dtu_scan*_v*')
              if os.path.exists(f'{d}/config.yaml') and glob.glob(f'{d}/seed_*/base.pt'))
cfgs = {d: yaml.safe_load(open(f'{d}/config.yaml', encoding='utf-8')) for d in dirs}
print(len(dirs), 'run dir,', sum(len(glob.glob(f'{d}/seed_*/base.pt')) for d in dirs), 'checkpoint')
for root in sorted({c['scene']['root'] for c in cfgs.values()}):     # tạo cache bề mặt tham chiếu lọc chặt
    c = next(c for c in cfgs.values() if c['scene']['root'] == root)
    _, _, ref = load_eval_scene(c, 'cpu', 'strict', 0.8, 16)
    print(root, 'strict ref:', len(ref), 'điểm')

In [ ]:
def eval_parallel(dirs, tag, extra, workers):
    groups = {}
    for d in dirs:
        sc = cfgs[d]['scene']; groups.setdefault((sc['root'], sc['n_train_views']), []).append(d)
    keys = sorted(groups, key=lambda k: -len(groups[k]))
    buckets = [[] for _ in range(min(workers, len(keys)))]
    for i, k in enumerate(keys):
        min(buckets, key=len).extend(groups[k])
    procs = []
    for i, b in enumerate(buckets):
        out = f'{RES}/{tag}_part{i}.csv'
        log = open(f'{RES}/{tag}_part{i}.log', 'w')
        procs.append((subprocess.Popen([sys.executable, '-W', 'ignore', 'scripts/floater_metrics.py', '--run_dirs', *b,
                                        '--seeds', '0', '1', '2', '3', '4', '--out', out, *extra],
                                       stdout=log, stderr=subprocess.STDOUT), out, log))
    t0 = time.time()
    for pr, out, log in procs:
        pr.wait(); log.close()
        print(f'{out}: exit {pr.returncode} ({(time.time() - t0) / 60:.1f} phút)', flush=True)
    df = pd.concat([pd.read_csv(o) for _, o, _ in procs if os.path.exists(o)], ignore_index=True)
    df.to_csv(f'{RES}/{tag}.csv', index=False)
    return df

EVAL_WORKERS = max(1, min(WORKERS, 12))
final = eval_parallel(dirs, 'final_strict_pool16', ['--ref_mode', 'strict', '--test_pool', '16'], EVAL_WORKERS)
import re
base8 = [d for d in dirs if re.fullmatch(r'runs/photo_(none|mild|strong)_dtu_scan\d+_v8', d)]
old = eval_parallel(base8, 'default_pool16', ['--ref_mode', 'default', '--test_pool', '16'], EVAL_WORKERS)
print(len(final), 'dòng (strict);', len(old), 'dòng (default)')

## 6. Phân tích và kết luận

Quy ước: `ΔW` = W_float(0.1) của một mức nhiễu trừ mức none **cùng scan, seed, N và cùng biến thể** (biến thể không có
mức none riêng, như nhiễu toàn cục, theo pixel, oracle, dùng mức none của biến thể gốc). `up` = số cặp tăng / tổng số cặp.

In [ ]:
import re
df = pd.read_csv(f'{RES}/final_strict_pool16.csv')
def level(r):
    if r.exposure_stops == 0: return 'none'
    s = 'mild' if r.exposure_stops == 0.5 else 'strong'
    if r.gamma == 0: s += '_nogamma'
    if r.photo_seed != 0: s += f'_noise{int(r.photo_seed)}'
    return s
def variant(r):
    v = []
    if r.photo_mode != 'isp': v.append(r.photo_mode)
    if r.appearance != 'none': v.append(r.appearance + ('_lr1e-2' if r.appearance == 'affine' and r.lr_appearance > 5e-3 else ''))
    if r.sh_degree: v.append(f'sh{int(r.sh_degree)}')
    if r.cap: v.append('cap')
    if r.init != 'random': v.append(r.init)
    return '+'.join(v) or 'base'
def base_variant(v):   # biến thể dùng làm mức none khi không có none riêng
    parts = [p for p in v.split('+') if p not in ('pixel', 'global', 'oracle')]
    return '+'.join(parts) or 'base'
df['level'] = df.apply(level, axis=1); df['variant'] = df.apply(variant, axis=1)
none = df[df.level == 'none'].set_index(['variant', 'scan', 'N', 'seed'])['wfloat_01']
def dW(row):
    for v in (row.variant, base_variant(row.variant)):
        k = (v, row.scan, row.N, row.seed)
        if k in none.index: return row.wfloat_01 - none[k]
    return np.nan
df['dW'] = df.apply(dW, axis=1)
df.to_csv(f'{RES}/final_with_delta.csv', index=False)

def table(sub, by=('variant', 'scan', 'N', 'level')):
    g = sub.groupby(list(by))
    t = g.agg(W=('wfloat_01', 'mean'), dW=('dW', 'mean'), sd=('dW', 'std'), n=('seed', 'count'),
              up=('dW', lambda x: f'{(x > 0).sum()}/{x.notna().sum()}'), psnr=('psnr', 'mean'),
              acc=('acc', 'mean'), comp=('comp', 'mean'))
    return t.round(3)
pd.set_option('display.width', 200); pd.set_option('display.max_rows', 500)
print(table(df).to_string())

In [ ]:
# P1 trên mọi scan, N = 8, biến thể gốc; và ảnh hưởng của việc lọc bề mặt tham chiếu
b = df[(df.variant == 'base') & (df.N == 8)]
print('P1 (strict ref, pool16):\n', table(b, ('scan', 'level')).to_string())
d = b[b.level.isin(['mild', 'strong'])]
print(f"\nTổng: {int((d.dW > 0).sum())}/{d.dW.notna().sum()} cặp tăng; mild {d[d.level=='mild'].dW.mean():+.3f}, "
      f"strong {d[d.level=='strong'].dW.mean():+.3f}")
o = pd.read_csv(f'{RES}/default_pool16.csv'); o['level'] = o.apply(level, axis=1)
cmp = (b.groupby(['scan', 'level']).wfloat_01.mean().rename('W_strict').to_frame()
       .join(o.groupby(['scan', 'level']).wfloat_01.mean().rename('W_default')))
print('\nW_float: bề mặt tham chiếu lọc chặt vs cũ (cùng view kiểm định):\n', cmp.round(3).to_string())

In [ ]:
# Cơ chế: E1 (pixel), E1b (toàn cục), E4 (affine), E4b (oracle, không gamma), E4c (affine lr 1e-2), E5, E6, M1 — N = 8, scan63/83
m = df[(df.N == 8) & df.scan.isin(['scan63', 'scan83']) & (df.init == 'random')]
piv = m[m.level != 'none'].pivot_table(index=['scan', 'level'], columns='variant', values='dW', aggfunc='mean')
print('ΔW so với none (N = 8):\n', piv.round(3).to_string())
W = m.pivot_table(index=['scan', 'level'], columns='variant', values='wfloat_01', aggfunc='mean')
print('\nW_float tuyệt đối:\n', W.round(3).to_string())
P = m.pivot_table(index=['scan', 'level'], columns='variant', values='psnr', aggfunc='mean')
print('\nPSNR kiểm định:\n', P.round(2).to_string())

def ratio(v, lvl='strong', ref='base'):
    a = piv.xs(lvl, level='level')
    return (a[v] / a[ref]).round(2).to_dict() if v in a and ref in a else None
print('\n--- Tóm tắt (tỉ lệ ΔW so với biến thể gốc, mức strong) ---')
for v in ['pixel', 'global', 'sh3', 'cap', 'affine', 'affine_lr1e-2', 'oracle', 'cf_log', 'cf_lin']:
    print(f'{v:15s}', ratio(v))
if 'strong_nogamma' in piv.index.get_level_values('level'):
    a = piv.xs('strong_nogamma', level='level')
    print('không gamma: ΔW gốc', a.get('base', pd.Series()).round(3).to_dict(), '| affine', a.get('affine', pd.Series()).round(3).to_dict())

In [ ]:
# Tương tác N x S: biến thể gốc (khởi tạo ngẫu nhiên) N = 4..16 và visual hull N = 4, 8. Hồi quy logit + bootstrap theo (scan, seed).
def interaction(sub, name, B=2000):
    sub = sub[sub.level.isin(['none', 'mild', 'strong'])].copy()
    sub['S'] = sub.level.map({'none': 0., 'mild': .5, 'strong': 1.})
    y = np.log(sub.wfloat_01.clip(1e-4, 1 - 1e-4) / (1 - sub.wfloat_01.clip(1e-4, 1 - 1e-4))).to_numpy()
    scans = sorted(sub.scan.unique())
    X = np.column_stack([np.ones(len(sub)), sub.N, sub.S, sub.N * sub.S] + [(sub.scan == s).astype(float) for s in scans[1:]])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    cl = (sub.scan + '_' + sub.seed.astype(str)).to_numpy(); uc = np.unique(cl)
    rng, bs = np.random.default_rng(0), []
    for _ in range(B):
        idx = np.concatenate([np.flatnonzero(cl == c) for c in rng.choice(uc, len(uc))])
        bs.append(np.linalg.lstsq(X[idx], y[idx], rcond=None)[0][:4])
    lo, hi = np.percentile(bs, [2.5, 97.5], axis=0)
    print(f'\n[{name}] N ∈ {sorted(sub.N.unique())}, {len(sub)} lượt')
    for nm, b_, l_, h_ in zip(['b0', 'N', 'S', 'N*S'], beta, lo, hi):
        print(f'  {nm:4s} {b_:+.4f}  CI95 [{l_:+.4f}, {h_:+.4f}]')
    for N in sorted(sub.N.unique()):
        print(f'  tác động của S tại N={N:2d} (logit): {beta[2] + beta[3] * N:+.3f}')
    t = sub[sub.level != 'none'].pivot_table(index=['scan', 'level'], columns='N', values='dW', aggfunc='mean')
    print(t.round(3).to_string())
    return beta
r = df[df.scan.isin(['scan63', 'scan83'])]
interaction(r[r.variant == 'base'], 'khởi tạo ngẫu nhiên')
interaction(r[(r.variant == 'base') & (r.N <= 8)], 'khởi tạo ngẫu nhiên, N ≤ 8')
if (r.variant == 'hull').any():
    interaction(r[r.variant == 'hull'], 'visual hull')

In [ ]:
# E9 (khởi tạo), E8 (cấu hình nhiễu), E10 (scan khác)
print('E9 – visual hull vs ngẫu nhiên:\n',
      df[df.variant.isin(['base', 'hull']) & df.N.isin([4, 8]) & df.level.isin(L3)]
        .pivot_table(index=['scan', 'N', 'level'], columns='variant', values=['wfloat_01', 'dW'], aggfunc='mean').round(3).to_string())
e8 = df[(df.variant == 'base') & (df.N == 8) & df.level.str.startswith('strong') & ~df.level.str.contains('nogamma')]
print('\nE8 – ΔW mức strong theo cấu hình nhiễu:\n', e8.pivot_table(index='scan', columns='level', values='dW', aggfunc='mean').round(3).to_string())
print('\nE11 – mức none theo seed (dao động):\n',
      df[(df.N == 8) & (df.level == 'none') & df.variant.isin(['base', 'affine'])]
        .pivot_table(index=['scan', 'variant'], columns='seed', values='wfloat_01').round(3).to_string())

### E7 – tín hiệu phát hiện floater (phân tích trên checkpoint, không huấn luyện)

Với mỗi Gaussian của các run gốc N = 8: độ mờ, kích thước, số view huấn luyện quan sát (`n_vis`), và **độ bất đồng giữa
hai seed** (khoảng cách tới Gaussian gần nhất của mô hình seed khác; dạng đơn giản của point disagreement trong CoR-GS,
không đồng huấn luyện). Nhãn: khoảng cách tới bề mặt tham chiếu lọc chặt > 0.1. AUROC có trọng số theo đóng góp trên view kiểm định.
Câu hỏi: tín hiệu nào còn phát hiện được floater khi có sai lệch trắc quang (đầu vào cho RQ3 / method).

In [ ]:
from sklearn.metrics import roc_auc_score
from frecon.gaussians import nearest_dist
from frecon.render import get_rasterizer
from frecon.train import TrainConfig, Trainer
dev = 'cuda'
rows = []
for s in ('63', '83'):
    for l in L3:
        rd = f'runs/photo_{l}_dtu_scan{s}_v8'
        cfg = yaml.safe_load(open(f'{rd}/config.yaml', encoding='utf-8'))
        ras, bg = get_rasterizer('torch'), torch.ones(3, device=dev)
        views, test_cams, ref = load_eval_scene(cfg, dev, 'strict', 0.8, 16)
        models = {k: Trainer.load(f'{rd}/seed_{k}/base.pt', TrainConfig(**cfg['train']), ras, bg, dev).model for k in SEEDS}
        for k, m in models.items():
            with torch.no_grad():
                d = nearest_dist(m.means.detach(), ref); lab = (d > 0.1).cpu().numpy()
                held = sum(m.render(ras, c, bg, return_contrib=True)['contrib'] for c in test_cams)
                nvis = sum((m.render(ras, c, bg, return_contrib=True)['contrib'] > 1).float() for c, _, _ in views)
                o = models[(k + 1) % len(SEEDS)]
                dis = nearest_dist(m.means.detach(), o.means.detach()[o.opacities.detach().reshape(-1) > 0.05])
                feats = {'opacity (thấp)': -m.opacities.detach().reshape(-1), 'kích thước': m.scales.max(1).values,
                         'n_vis (thấp)': -nvis, 'bất đồng 2 seed': dis}
                w = held.cpu().numpy() + 1e-9
                for name, f in feats.items():
                    rows.append({'scan': f'scan{s}', 'level': l, 'seed': k, 'signal': name,
                                 'auroc_w': roc_auc_score(lab, f.cpu().numpy(), sample_weight=w),
                                 'auroc': roc_auc_score(lab, f.cpu().numpy())})
        del models; torch.cuda.empty_cache()
        print(f'scan{s} {l} xong', flush=True)
e7 = pd.DataFrame(rows); e7.to_csv(f'{RES}/E7_signals.csv', index=False)
print(e7.pivot_table(index=['signal'], columns=['scan', 'level'], values='auroc_w').reindex(columns=L3, level='level').round(3).to_string())

In [ ]:
!ls -la {RES}